# Replace videoVisual with PaliGemma Emotion BERT Embeddings
Encodes the `paligemma_emotion` column from `manifest.csv` (single-word emotion label per utterance) using `bert-large-uncased` (CLS token) and saves a new pkl where `videoVisual` is replaced with the **1024-dim** embeddings.

In [1]:
import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm

# Paths
PKL_IN   = "Dataset/CFN-ESA/iemocap_multi_features.pkl"
PKL_OUT  = "Dataset/CFN-ESA/iemocap_vlm_visual.pkl"
MANIFEST = "manifest.csv"
MODEL_NAME = "bert-large-uncased"
BATCH_SIZE = 128   # single-word inputs are tiny — large batches are fine

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/mnt/Work/Environments/Ubuntu/Conda/envs/ml/lib/python3.11/site-packages/torch/cuda/__init__.py:58: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device : cuda
GPU    : NVIDIA GeForce RTX 3060
VRAM   : 12.5 GB


/mnt/Work/Environments/Ubuntu/Conda/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Load the original PKL and the manifest

In [2]:
# ── Load original pkl ────────────────────────────────────────────────────────
(
    videoIDs, videoSpeakers, videoLabels,
    videoText0, videoText1, videoText2, videoText3,
    videoAudio, videoVisual,
    videoSentence, trainVid, testVid,
) = pickle.load(open(PKL_IN, "rb"), encoding="latin1")

all_vids = sorted(videoIDs.keys())
total_utts = sum(len(videoIDs[v]) for v in all_vids)
print(f"Dialogues loaded : {len(all_vids)}")
print(f"Total utterances : {total_utts}")
print(f"Original visual dim : {np.array(videoVisual[all_vids[0]]).shape[1]}")

# ── Load manifest ─────────────────────────────────────────────────────────────
df = pd.read_csv(MANIFEST)
print(f"\nManifest rows : {len(df)}")
print(f"Columns       : {list(df.columns)}")

# Build fast lookup: utterance_id → paligemma_emotion (single word)
vlm_lookup = dict(zip(df["utterance_id"], df["paligemma_emotion"]))
print(f"Lookup entries : {len(vlm_lookup)}")
print(f"Unique emotion words: {sorted(df['paligemma_emotion'].unique())}")

Dialogues loaded : 151
Total utterances : 7433
Original visual dim : 342

Manifest rows : 10039
Columns       : ['utterance_id', 'emotion', 'speaker_gender', 'crop_side', 'start', 'end', 'path', 'vlm_analysis', 'paligemma_emotion']
Lookup entries : 10039
Unique emotion words: ['anger', 'contempt', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


## Step 2 — Coverage check: which pkl utterances are in the manifest?

In [3]:
# Collect every utterance_id present in the pkl
all_pkl_utts = [uid for v in all_vids for uid in videoIDs[v]]

missing = [uid for uid in all_pkl_utts if uid not in vlm_lookup]
found   = [uid for uid in all_pkl_utts if uid in vlm_lookup]

print(f"PKL utterances          : {len(all_pkl_utts)}")
print(f"Found in manifest       : {len(found)}  ({100*len(found)/len(all_pkl_utts):.1f}%)")
print(f"Missing from manifest   : {len(missing)}")

if missing:
    raise KeyError(f"{len(missing)} utterances not found in manifest:\n" +
                   "\n".join(f"  {uid}" for uid in missing[:20]))

PKL utterances          : 7433
Found in manifest       : 7433  (100.0%)
Missing from manifest   : 0


## Step 3 — Load BERT-large

In [4]:
from transformers import AutoTokenizer, AutoModel

print(f"Loading tokenizer and model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, use_safetensors=True)
model.eval()
model.to(DEVICE)

hidden_size = model.config.hidden_size
print(f"Model hidden size (→ new visual dim): {hidden_size}")
print("Model loaded successfully.")

Loading tokenizer and model: bert-large-uncased


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1352.96it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-large-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model hidden size (→ new visual dim): 1024
Model loaded successfully.


## Step 4 — Encode all utterances with BERT (single-word inputs, batched)

In [5]:
def encode_texts_bert(texts, tokenizer, model, device, batch_size=128):
    """
    Simple batched BERT encoding.
    Each text is tokenized to [CLS] <tokens> [SEP] and the CLS hidden state
    at position 0 is used as the embedding.
    Returns np.ndarray of shape (len(texts), hidden_size), dtype float32.
    """
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=False, return_tensors="pt")
        input_ids      = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=attention_mask)

        cls = out.last_hidden_state[:, 0, :].cpu().float().numpy()  # (batch, 1024)
        all_embeddings.append(cls)

    return np.concatenate(all_embeddings, axis=0)  # (N, 1024)


# ── Build flat list ───────────────────────────────────────────────────────────
flat_records = []   # (vid, utt_idx, utt_id)
flat_texts   = []   # parallel paligemma_emotion words

for vid in all_vids:
    for utt_idx, utt_id in enumerate(videoIDs[vid]):
        word = vlm_lookup[utt_id]   # KeyError if missing — no silent fallbacks
        flat_records.append((vid, utt_idx, utt_id))
        flat_texts.append(word)

print(f"Total utterances   : {len(flat_texts)}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Batches            : {(len(flat_texts) + BATCH_SIZE - 1) // BATCH_SIZE}")

# Token lengths (should all be 3: [CLS] word [SEP])
sample_enc = tokenizer(flat_texts[:10], return_tensors="pt", padding=True)
print(f"\nSample token shapes: {sample_enc['input_ids'].shape}")
print(f"Sample tokens (first 3 words):")
for word, ids in zip(flat_texts[:3], sample_enc['input_ids'][:3]):
    decoded = tokenizer.convert_ids_to_tokens(ids.tolist())
    print(f"  '{word}' → {decoded}")

Total utterances   : 7433
Batch size         : 128
Batches            : 59

Sample token shapes: torch.Size([10, 3])
Sample tokens (first 3 words):
  'neutral' → ['[CLS]', 'neutral', '[SEP]']
  'neutral' → ['[CLS]', 'neutral', '[SEP]']
  'neutral' → ['[CLS]', 'neutral', '[SEP]']


In [6]:
print("Starting encoding...")
all_embeddings = encode_texts_bert(flat_texts, tokenizer, model, DEVICE, batch_size=BATCH_SIZE)
print(f"\nEncoding complete.")
print(f"Output shape : {all_embeddings.shape}  (expected: {len(flat_texts)} × {hidden_size})")
print(f"dtype        : {all_embeddings.dtype}")
print(f"value range  : [{all_embeddings.min():.4f}, {all_embeddings.max():.4f}]")

Starting encoding...


Encoding: 100%|██████████| 59/59 [00:03<00:00, 17.54it/s]



Encoding complete.
Output shape : (7433, 1024)  (expected: 7433 × 1024)
dtype        : float32
value range  : [-2.4723, 5.0992]


## Step 5 — Rebuild `videoVisual` and save new pkl

In [7]:
# ── Rebuild videoVisual dict ──────────────────────────────────────────────────
# flat_records[i] = (vid, utt_idx, utt_id)
# all_embeddings[i] = 768-dim CLS embedding

new_videoVisual = {vid: [None] * len(videoIDs[vid]) for vid in all_vids}

for i, (vid, utt_idx, utt_id) in enumerate(flat_records):
    new_videoVisual[vid][utt_idx] = all_embeddings[i]   # np.float32 (768,)

# Convert lists to arrays for consistency with the original pkl format
for vid in all_vids:
    new_videoVisual[vid] = np.array(new_videoVisual[vid], dtype=np.float32)

print("new_videoVisual sanity check:")
sample = all_vids[0]
arr = new_videoVisual[sample]
print(f"  [{sample}]  shape={arr.shape}  dtype={arr.dtype}")
print(f"  Original : {np.array(videoVisual[sample]).shape}")
assert arr.shape == (len(videoIDs[sample]), hidden_size), "Shape mismatch!"

# ── Reconstruct the 12-field pkl tuple ───────────────────────────────────────
# Original field order (matches IEMOCAPDataset_BERT unpacking):
#  0: videoIDs  1: videoSpeakers  2: videoLabels
#  3: videoText0  4: videoText1  5: videoText2  6: videoText3
#  7: videoAudio  8: videoVisual  9: videoSentence  10: trainVid  11: testVid

new_data = [
    videoIDs, videoSpeakers, videoLabels,
    videoText0, videoText1, videoText2, videoText3,
    videoAudio,
    new_videoVisual,        # ← replaced
    videoSentence, trainVid, testVid,
]

# ── Save ─────────────────────────────────────────────────────────────────────
Path(PKL_OUT).parent.mkdir(parents=True, exist_ok=True)
with open(PKL_OUT, "wb") as f:
    pickle.dump(new_data, f, protocol=4)

size_mb = Path(PKL_OUT).stat().st_size / 1e6
print(f"\nSaved to  : {PKL_OUT}")
print(f"File size : {size_mb:.1f} MB")
print("Done.")

new_videoVisual sanity check:
  [Ses01F_impro01]  shape=(26, 1024)  dtype=float32
  Original : (26, 342)

Saved to  : Dataset/CFN-ESA/iemocap_vlm_visual.pkl
File size : 201.4 MB
Done.


## Step 6 — Verify the new pkl by reloading from disk

In [8]:
# Reload from disk and run full assertions
print(f"Reloading  : {PKL_OUT}")
(
    v_videoIDs, v_videoSpeakers, v_videoLabels,
    v_videoText0, v_videoText1, v_videoText2, v_videoText3,
    v_videoAudio, v_videoVisual_new,
    v_videoSentence, v_trainVid, v_testVid,
) = pickle.load(open(PKL_OUT, "rb"), encoding="latin1")

# ── Structural checks ─────────────────────────────────────────────────────────
assert len(v_videoIDs) == 151,   f"Expected 151 dialogues, got {len(v_videoIDs)}"
assert v_trainVid == trainVid,   "trainVid mismatch"
assert v_testVid  == testVid,    "testVid mismatch"

total_new = sum(len(v_videoIDs[v]) for v in sorted(v_videoIDs))
assert total_new == 7433, f"Expected 7433 utterances, got {total_new}"

# ── Visual dim check ──────────────────────────────────────────────────────────
for vid in sorted(v_videoIDs):
    arr = np.array(v_videoVisual_new[vid])
    expected_len = len(v_videoIDs[vid])
    assert arr.shape == (expected_len, hidden_size), \
        f"{vid}: expected ({expected_len}, {hidden_size}), got {arr.shape}"

# ── Other fields unchanged ────────────────────────────────────────────────────
sample = all_vids[0]
assert np.allclose(np.array(v_videoText0[sample]), np.array(videoText0[sample])), "Text0 changed!"
assert np.allclose(np.array(v_videoAudio[sample]),  np.array(videoAudio[sample])),  "Audio changed!"
assert v_videoLabels[sample] == videoLabels[sample], "Labels changed!"

print("All assertions passed.\n")
print(f"  Dialogues     : {len(v_videoIDs)}")
print(f"  Utterances    : {total_new}")
print(f"  New visual dim: {np.array(v_videoVisual_new[all_vids[0]]).shape[1]}  (was 342)")
print(f"  Audio dim     : {np.array(v_videoAudio[all_vids[0]]).shape[1]}  (unchanged)")
print(f"  Text dim      : {np.array(v_videoText0[all_vids[0]]).shape[1]}  (unchanged)")
print(f"\nSample dialogue '{sample}':")
print(f"  old visual shape : {np.array(videoVisual[sample]).shape}")
print(f"  new visual shape : {np.array(v_videoVisual_new[sample]).shape}")
print(f"\nPKL is ready for training → {PKL_OUT}")

Reloading  : Dataset/CFN-ESA/iemocap_vlm_visual.pkl
All assertions passed.

  Dialogues     : 151
  Utterances    : 7433
  New visual dim: 1024  (was 342)
  Audio dim     : 1582  (unchanged)
  Text dim      : 1024  (unchanged)

Sample dialogue 'Ses01F_impro01':
  old visual shape : (26, 342)
  new visual shape : (26, 1024)

PKL is ready for training → Dataset/CFN-ESA/iemocap_vlm_visual.pkl
